# 07 - Earth Engine extractions (Python API, in-process)

Runs the three remaining Earth Engine jobs **inside this notebook**, the same way notebook 02
already works: no Code Editor, no Drive round-trip. Each section writes its CSV straight into
`data/`, where notebooks 05 and 06 pick it up.

| Section | Output | Feeds |
|---|---|---|
| 1. MODIS cloud statistics | `data/MODIS_cloud_statistics.csv` | notebook 05 · manuscript Section 2.5 |
| 2. Daily melt timing | `data/MODIS_melt_timing.csv` | notebook 06 · manuscript Section 3.9 |
| 3. Hydrology + GRACE | `data/ERA5Land_KRI_hydrology.csv`, `data/GRACE_KRI_TWS.csv` | notebook 06 · manuscript Section 3.10 |

**Every section is resumable.** Results are appended to the CSV as each chunk completes, and
re-running the cell skips the chunks already present. If a request times out, just run the cell
again — it continues from where it stopped.

The equivalent Code Editor scripts remain in `gee/` if you ever prefer that route.

In [1]:
import os, time
import numpy as np, pandas as pd
import ee

ROOT   = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA   = os.path.join(ROOT, "data", "derived")
os.makedirs(DATA, exist_ok=True)

EE_PROJECT = "your-earthengine-project-id"
ASSET_ID   = "projects/your-earthengine-project-id/assets/mountain_Region_KRI"
SHP        = os.path.join(ROOT, "data", "raw", "mountain_Region_KRI.shp")

ee.Initialize(project=EE_PROJECT)

# The uploaded asset is preferred: the geometry stays server-side, which is much faster than
# sending a detailed polygon with every request. Falls back to the shapefile if the asset is
# unavailable.
try:
    region = ee.FeatureCollection(ASSET_ID).geometry()
    area = ee.Number(region.area(1000)).divide(1e6).getInfo()
    print(f"region from asset: {area:,.1f} km²")
except Exception as e:
    print("asset unavailable ->", type(e).__name__, e)
    import geopandas as gpd, json as _json
    gdf  = gpd.read_file(SHP).to_crs(4326)
    geom = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
    region = ee.Geometry(_json.loads(gpd.GeoSeries([geom], crs=4326).to_json())
                         ["features"][0]["geometry"])
    print(f"region from shapefile: {ee.Number(region.area(1000)).divide(1e6).getInfo():,.1f} km²")

region from asset: 26,165.5 km²


In [2]:
def fc_to_df(fc):
    """FeatureCollection -> DataFrame of its properties."""
    return pd.DataFrame([f["properties"] for f in fc.getInfo()["features"]])

def append_csv(path, df, key_cols):
    """Append rows to a CSV, dropping duplicates on key_cols. Makes every section resumable."""
    if os.path.exists(path):
        old = pd.read_csv(path)
        df = pd.concat([old, df], ignore_index=True)
    df = df.drop_duplicates(subset=key_cols, keep="last").sort_values(key_cols)
    df.to_csv(path, index=False)
    return df

def done_keys(path, key_cols):
    if not os.path.exists(path):
        return set()
    d = pd.read_csv(path)
    return set(map(tuple, d[key_cols].values.tolist()))

SEASONS = list(range(2000, 2024))          # season 2000 = Oct 2000 - Mar/May 2001
print("ready")

ready


## 1. MODIS data-availability (cloud) statistics

For every month of the Oct–Mar seasons, counts per pixel how many daily MOD10A1 granules
carried a usable snow retrieval and how many did not, then takes the area mean.

**What this can and cannot measure.** Earth Engine's MOD10A1 ingestion *masks* the
non-retrieval class codes instead of storing them as values, so the cloud code — 250 in
`NDSI_Snow_Cover`, 150 in `Snow_Albedo_Daily_Tile` — is never present to be counted, and any
test for it returns zero everywhere. Cloud therefore cannot be separated from polar night and
missing granules using these bands. What *can* be measured is the combined data gap, and in
midwinter that gap is overwhelmingly cloud. `gap_fraction` is the number to quote; describe it
in the manuscript as "cloud, night or missing", not as cloud alone.

**One more trap:** Earth Engine masks unavailable pixel-days and `sum()` skips masked values,
so the days being counted must be `unmask(0)`'d first or the sums silently return zero.

The most useful output is not the mean gap itself but its **trend**: a non-significant trend
shows the snow-cover decline is not an artefact of changing data availability, which is the
sharper version of what the reviewer asked for.

144 requests, roughly 2–5 minutes.

In [3]:
CLOUD_CSV = os.path.join(DATA, "MODIS_cloud_statistics.csv")
SCALE_MODIS = 500

mod = ee.ImageCollection("MODIS/061/MOD10A1")
print("MOD10A1 bands:", mod.first().bandNames().getInfo())

# What can and cannot be measured here
# ------------------------------------
# Earth Engine's MOD10A1 ingestion masks the non-retrieval class codes rather than storing
# them as values, so the cloud code (250 in NDSI_Snow_Cover, 150 in Snow_Albedo_Daily_Tile)
# is never present to be counted - any test for it returns zero everywhere. Cloud therefore
# cannot be separated from polar night and missing granules with these bands.
# What CAN be measured, and what this cell reports, is the combined data gap: the share of
# available daily granules for which a pixel has no usable snow retrieval. In midwinter that
# gap is overwhelmingly cloud, and it is the honest number to quote.
#
# Note also: Earth Engine masks unavailable pixel-days and sum() skips masked values, so the
# days being counted must be unmask(0)'d first or the sums silently return zero.


def cloud_month(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    sub   = mod.filterDate(start, start.advance(1, "month"))
    n     = ee.Number(sub.size())
    n_img = ee.Image.constant(n).toFloat()

    valid = (sub.map(lambda i: i.select("NDSI_Snow_Cover").lte(100).unmask(0))
                .sum().toFloat().rename("valid_days"))
    gap   = n_img.subtract(valid).rename("gap_days")
    img   = (valid.addBands(gap)
                  .addBands(gap.divide(n_img.max(1)).rename("gap_fraction")))

    stats = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=region,
                             scale=SCALE_MODIS, maxPixels=int(1e13),
                             bestEffort=True, tileScale=4)
    return ee.Dictionary(stats).combine({"year": year, "month": month, "n_images": n})


todo = [(s + (0 if m >= 10 else 1), m) for s in SEASONS for m in (10, 11, 12, 1, 2, 3)]
print(f"computing {len(todo)} months (previous results are overwritten)")

t0, buf = time.time(), []
for k, (yr, mo) in enumerate(todo, 1):
    try:
        buf.append(cloud_month(yr, mo).getInfo())
    except Exception as e:
        print(f"  {yr}-{mo:02d} failed ({type(e).__name__}: {str(e)[:80]}) - re-run to retry")
        continue
    if k % 12 == 0 or k == len(todo):
        append_csv(CLOUD_CSV, pd.DataFrame(buf), ["year", "month"]); buf = []
        print(f"  {k}/{len(todo)} done ({time.time()-t0:.0f}s)")
if buf:
    append_csv(CLOUD_CSV, pd.DataFrame(buf), ["year", "month"])

cl = pd.read_csv(CLOUD_CSV)
# drop the all-zero cloud columns written by the earlier version of this cell
for c in ["cloud_days", "cloud_fraction"]:
    if c in cl.columns and (cl[c].fillna(0) == 0).all():
        cl = cl.drop(columns=c)
cl.to_csv(CLOUD_CSV, index=False)

assert np.allclose(cl.valid_days + cl.gap_days, cl.n_images, atol=0.01), \
    "valid_days + gap_days must equal the number of available granules"
print(f"\nsaved {CLOUD_CSV}  ({len(cl)} rows)")
print(f"usable daily retrievals per pixel-month : {cl['valid_days'].mean():.1f} "
      f"of {cl['n_images'].mean():.1f} available days")
print(f"unusable (cloud, night, missing)        : {cl['gap_fraction'].mean()*100:.1f}%")
cl.head()

MOD10A1 bands: ['NDSI_Snow_Cover', 'NDSI_Snow_Cover_Class', 'NDSI_Snow_Cover_Basic_QA', 'NDSI_Snow_Cover_Algorithm_Flags_QA', 'NDSI', 'Snow_Albedo_Daily_Tile', 'Snow_Albedo_Daily_Tile_Class', 'orbit_pnt', 'granule_pnt']
computing 144 months (previous results are overwritten)
  12/144 done (21s)
  24/144 done (38s)
  36/144 done (57s)
  48/144 done (76s)
  60/144 done (95s)
  72/144 done (112s)
  84/144 done (131s)
  96/144 done (152s)
  108/144 done (176s)
  120/144 done (195s)
  132/144 done (213s)
  144/144 done (228s)

saved <repo>/data\MODIS_cloud_statistics.csv  (144 rows)
usable daily retrievals per pixel-month : 15.2 of 30.1 available days
unusable (cloud, night, missing)        : 49.5%


,month,n_images,valid_days,year,gap_days,gap_fraction
0,10,31,19.381891,2000,11.618109,0.374778
1,11,30,19.890629,2000,10.109371,0.336979
2,12,31,10.140603,2000,20.859397,0.672884
3,1,31,14.944208,2001,16.055792,0.517929
4,2,28,9.340961,2001,18.659039,0.666394


## 2. Snowmelt timing from daily MOD10A1

This is the analysis that monthly composites cannot deliver. For each season and each pixel it
derives, in days counted from 1 October:

- `scd_days` — snow cover duration, days observed as snow-covered. **This is the primary
  metric**: it is a plain count, so it is far more robust than either date, and it is what the
  comparable literature reports.
- `obs_days` — days actually observed at all, the denominator needed to read SCD honestly.
- `onset_doy`, `meltout_doy` — first and last snow day.

**Why the first version failed.** The error *"Expected a homogeneous image collection"* means
some images in a collection differ in band name or data type from the others. It came from
building the day-of-season images by arithmetic on a boolean band, which leaves the type
ambiguous. Every derived image is now explicitly `.rename()`d and `.toFloat()`ed, and the date
images are built with `ee.Image.constant`, which sidesteps the problem entirely.

**Persistence filter.** `USE_PERSISTENCE = False` by default, which gives the raw first and last
snow day. Set it to `True` to require 3 snow days within the 5 days centred on each date, which
stops a single cloud-edge misclassification from setting the onset date. That path uses a
`saveAll` join and is heavier; run the default first, and treat the filtered version as a
robustness check on the dates. `scd_days` is unaffected by the choice in any meaningful way.

One request per season, each spanning ~240 daily images: expect 1–3 minutes per season at
500 m. If requests time out, set `SCALE_MELT = 1000` — these are area means, so the coarser
scale barely moves them. The cell is resumable, so a partial run simply continues.

In [4]:
MELT_CSV   = os.path.join(DATA, "MODIS_melt_timing.csv")
SCALE_MELT = 500          # raise to 1000 if requests time out
NDSI_THR   = 20           # NDSI * 100, matching the 0.20 threshold used in the paper

# Persistence filter: leave False for the robust run. See the note below the results.
USE_PERSISTENCE = False
WIN_DAYS, MIN_SNOW = 2, 3     # +/- 2 days (a 5-day window), 3 snow days required


def season_timing(y0):
    start = ee.Date.fromYMD(y0, 10, 1)
    end   = ee.Date.fromYMD(y0 + 1, 6, 1)

    raw = (ee.ImageCollection("MODIS/061/MOD10A1")
           .filterDate(start, end).select("NDSI_Snow_Cover"))

    def to_snow(img):
        # Every derived image gets the SAME band name and the SAME type. Earth Engine
        # rejects a collection whose images differ in either, which is what the
        # "Expected a homogeneous image collection" error means.
        snow = img.gte(NDSI_THR).And(img.lte(100)).rename("snow").toFloat()
        return snow.set({"doy": ee.Number(img.date().difference(start, "day")).add(1),
                         "system:time_start": img.get("system:time_start")})

    daily = raw.map(to_snow)

    if USE_PERSISTENCE:
        win  = ee.Filter.maxDifference(difference=WIN_DAYS * 86400000,
                                       leftField="system:time_start",
                                       rightField="system:time_start")
        joined = ee.ImageCollection(ee.Join.saveAll(matchesKey="win").apply(daily, daily, win))
        daily = joined.map(lambda img: (
            ee.ImageCollection.fromImages(img.get("win")).select("snow").sum()
              .gte(MIN_SNOW).rename("snow").toFloat()
              .set({"doy": img.get("doy"),
                    "system:time_start": img.get("system:time_start")})))

    # snow cover duration: sum() ignores masked (cloud/missing) days, so this is the number
    # of days observed as snow-covered, not the number of days assumed to be
    scd = daily.sum().rename("scd_days").toFloat()
    # how many days were actually observed at all - the denominator for interpreting SCD
    obs = (daily.map(lambda i: i.mask().rename("snow").toFloat())
                .sum().rename("obs_days").toFloat())

    def doy_image(i):
        # a fresh constant image per day, masked to the snow-covered pixels.
        # ee.Image.constant avoids the type ambiguity of arithmetic on a boolean band.
        return (ee.Image.constant(ee.Number(i.get("doy"))).toFloat().rename("doy")
                  .updateMask(i.select("snow")))

    doys    = daily.map(doy_image)
    onset   = doys.min().rename("onset_doy")
    meltout = doys.max().rename("meltout_doy")

    img = scd.addBands(obs).addBands(onset).addBands(meltout)
    stats = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=region,
                             scale=SCALE_MELT, maxPixels=int(1e13),
                             bestEffort=True, tileScale=4)
    return ee.Dictionary(stats).combine({"season_start": y0, "season_end": y0 + 1})


have = {k[0] for k in done_keys(MELT_CSV, ["season_start"])}
todo = [y for y in SEASONS if y not in have]
print(f"{len(todo)} season(s) to compute, {len(have)} already done "
      f"(persistence filter: {USE_PERSISTENCE})")

t0 = time.time()
for k, y0 in enumerate(todo, 1):
    try:
        row = season_timing(y0).getInfo()
    except Exception as e:
        print(f"  {y0}/{y0+1} failed ({type(e).__name__}: {str(e)[:110]})")
        continue
    append_csv(MELT_CSV, pd.DataFrame([row]), ["season_start"])
    print(f"  {y0}/{y0+1}: SCD {row.get('scd_days', float('nan')):5.1f} d of "
          f"{row.get('obs_days', float('nan')):5.1f} observed, "
          f"onset {row.get('onset_doy', float('nan')):5.1f}, "
          f"melt-out {row.get('meltout_doy', float('nan')):5.1f}   "
          f"[{k}/{len(todo)}, {time.time()-t0:.0f}s]")

if os.path.exists(MELT_CSV):
    mt = pd.read_csv(MELT_CSV)
    mt["season_label"] = (mt.season_start.astype(str) + "/" +
                          mt.season_end.astype(str).str[-2:])
    mt.to_csv(MELT_CSV, index=False)
    print(f"\nsaved {MELT_CSV}  ({len(mt)} seasons)")
    print(mt[["season_label", "scd_days", "obs_days", "onset_doy",
              "meltout_doy"]].round(1).to_string(index=False))

24 season(s) to compute, 0 already done (persistence filter: False)
  2000/2001: SCD   7.7 d of 130.4 observed, onset  95.1, melt-out 137.6   [1/24, 12s]
  2001/2002: SCD  11.5 d of 119.7 observed, onset  76.8, melt-out 133.3   [2/24, 18s]
  2002/2003: SCD  10.1 d of 111.8 observed, onset  80.4, melt-out 145.4   [3/24, 25s]
  2003/2004: SCD   6.3 d of 117.2 observed, onset 106.9, melt-out 154.0   [4/24, 33s]
  2004/2005: SCD  13.5 d of 129.2 observed, onset  86.6, melt-out 147.9   [5/24, 50s]
  2005/2006: SCD   8.7 d of 130.5 observed, onset  91.3, melt-out 144.4   [6/24, 60s]
  2006/2007: SCD  13.6 d of 121.7 observed, onset  73.2, melt-out 141.1   [7/24, 81s]
  2007/2008: SCD   9.9 d of 139.6 observed, onset  90.5, melt-out 137.8   [8/24, 93s]
  2008/2009: SCD   6.5 d of 129.0 observed, onset  87.8, melt-out 130.2   [9/24, 111s]
  2009/2010: SCD   3.5 d of 112.9 observed, onset  96.0, melt-out 138.7   [10/24, 120s]
  2010/2011: SCD   7.4 d of 143.3 observed, onset 100.6, melt-out 143

In [5]:
# quick trend check - the full version with figures lives in notebook 06
from scipy import stats as _st
if os.path.exists(MELT_CSV):
    mt = pd.read_csv(MELT_CSV)
    if len(mt) >= 10:
        print("Trends in melt timing (days from 1 October)\n")
        for col in ["onset_doy", "meltout_doy", "scd_days"]:
            d = mt[["season_start", col]].dropna()
            y, x = d[col].values, d["season_start"].values
            sl = np.median([(y[j]-y[i])/(x[j]-x[i])
                            for i in range(len(y)) for j in range(i+1, len(y))])
            tau, p = _st.kendalltau(x, y)
            print(f"{col:13s} mean = {y.mean():6.1f} d   Sen = {sl:+.3f} d/yr "
                  f"({sl*10:+.1f} d/decade)   tau = {tau:+.2f}, p = {p:.3f}")
    else:
        print(f"only {len(mt)} seasons so far - finish the extraction first")

Trends in melt timing (days from 1 October)

onset_doy     mean =   89.5 d   Sen = +0.157 d/yr (+1.6 d/decade)   tau = +0.13, p = 0.389
meltout_doy   mean =  140.1 d   Sen = -0.145 d/yr (-1.5 d/decade)   tau = -0.08, p = 0.606
scd_days      mean =    8.5 d   Sen = -0.147 d/yr (-1.5 d/decade)   tau = -0.23, p = 0.119


## 3. Hydrology: ERA5-Land runoff and GRACE water storage

ERA5-Land monthly snowmelt, runoff and evaporation over the study area, plus the GRACE/GRACE-FO
terrestrial water storage anomaly, which includes the groundwater component the reviewer
mentions.

**GRACE collection note.** `NASA/GRACE/MASS_GRIDS_V04/LAND` ends on 7 January 2017 despite its
description mentioning GRACE-FO, and its bands are per-processing-centre
(`lwe_thickness_csr`, `_gfz`, `_jpl`) rather than a plain `lwe_thickness` — which is why the
first version of this cell failed. The cell now uses the JPL mascon product
`NASA/GRACE/MASS_GRIDS_V04/MASCON`, which spans 2002–2024 including GRACE-FO, carries a single
`lwe_thickness` band in cm, and needs no de-striping or Gaussian smoothing.

Two caveats to carry into the text: the GRACE footprint (~300 km) is much larger than the study
area, so it represents the wider basin rather than the mountain block alone; and there is an
~11-month gap between the two missions in 2017–2018 that is simply absent from the imagery
rather than flagged.

ERA5-Land is fast; GRACE is one mapped request.

In [6]:
HYD_CSV   = os.path.join(DATA, "ERA5Land_KRI_hydrology.csv")
GRACE_CSV = os.path.join(DATA, "GRACE_KRI_TWS.csv")

HYDRO = ["snowmelt_sum", "runoff_sum", "surface_runoff_sum",
         "sub_surface_runoff_sum", "total_evaporation_sum",
         "snow_depth_water_equivalent"]

era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
        .filterDate("2000-01-01", "2024-10-01"))

def hyd_feature(img):
    s = img.select(HYDRO).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region,
        scale=1000, maxPixels=int(1e13), bestEffort=True, tileScale=4)
    return ee.Feature(None, s).set({"year": img.date().get("year"),
                                    "month": img.date().get("month")})

# split by year so no single request gets too large
frames = []
for yr in range(2000, 2025):
    sub = era5.filter(ee.Filter.calendarRange(yr, yr, "year"))
    try:
        d = fc_to_df(ee.FeatureCollection(sub.map(hyd_feature)))
        if len(d):
            frames.append(d)
            print(f"  {yr}: {len(d)} months")
    except Exception as e:
        print(f"  {yr} failed ({type(e).__name__}) - re-run to retry")

if frames:
    h = pd.concat(frames, ignore_index=True)
    h["date"] = (h.year.astype(int).astype(str) + "-" +
                 h.month.astype(int).astype(str).str.zfill(2))
    h = h[["date", "year", "month"] + [c for c in HYDRO if c in h]]
    append_csv(HYD_CSV, h, ["year", "month"])
    print(f"\nsaved {HYD_CSV}  ({len(pd.read_csv(HYD_CSV))} months)")

  2000: 12 months
  2001: 12 months
  2002: 12 months
  2003: 12 months
  2004: 12 months
  2005: 12 months
  2006: 12 months
  2007: 12 months
  2008: 12 months
  2009: 12 months
  2010: 12 months
  2011: 12 months
  2012: 12 months
  2013: 12 months
  2014: 12 months
  2015: 12 months
  2016: 12 months
  2017: 12 months
  2018: 12 months
  2019: 12 months
  2020: 12 months
  2021: 12 months
  2022: 12 months
  2023: 12 months
  2024: 9 months

saved <repo>/data\ERA5Land_KRI_hydrology.csv  (297 months)


In [7]:
# NASA/GRACE/MASS_GRIDS_V04/LAND ends on 2017-01-07 and its bands are per-centre
# (lwe_thickness_csr / _gfz / _jpl), so selecting "lwe_thickness" from it fails.
# The JPL mascon product is the one that spans GRACE *and* GRACE-FO.
GRACE_ID    = "NASA/GRACE/MASS_GRIDS_V04/MASCON"      # RL06.3Mv04 JPL mascons, 2002-2024
GRACE_SCALE = 55660                                    # native mascon resolution

grace = ee.ImageCollection(GRACE_ID).filterDate("2002-01-01", "2025-01-01")
print("GRACE bands:", grace.first().bandNames().getInfo())
print("images:", grace.size().getInfo())


def tws_feature(img):
    s = img.select("lwe_thickness").reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region,
        scale=GRACE_SCALE, maxPixels=int(1e13), bestEffort=True)
    return ee.Feature(None, s).set({"year": img.date().get("year"),
                                    "month": img.date().get("month")})

try:
    g = fc_to_df(ee.FeatureCollection(grace.map(tws_feature))).dropna(subset=["lwe_thickness"])
    # one image per solution period; average any duplicates within a calendar month
    g = g.groupby(["year", "month"], as_index=False)["lwe_thickness"].mean()
    g["date"] = (g.year.astype(int).astype(str) + "-" +
                 g.month.astype(int).astype(str).str.zfill(2))
    g = g[["date", "year", "month", "lwe_thickness"]].sort_values(["year", "month"])
    g.to_csv(GRACE_CSV, index=False)
    print(f"\nsaved {GRACE_CSV}  ({len(g)} months, {int(g.year.min())}-{int(g.year.max())})")
    missing = 12 * (g.year.max() - g.year.min() + 1) - len(g)
    print(f"months absent from the record: {missing} "
          "(includes the ~11-month gap between GRACE and GRACE-FO in 2017-2018)")
    print(g.tail(3).to_string(index=False))
except Exception as e:
    print("GRACE extraction failed:", type(e).__name__, e)
    print("Alternatives to try: 'NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI' (coastal-resolution "
          "filtered), or 'NASA/GRACE/MASS_GRIDS_V04/LAND' with band 'lwe_thickness_jpl' "
          "if you only need 2002-2017.")

GRACE bands: ['lwe_thickness', 'uncertainty']
images: 238

saved <repo>/data\GRACE_KRI_TWS.csv  (227 months, 2002-2024)
months absent from the record: 49 (includes the ~11-month gap between GRACE and GRACE-FO in 2017-2018)
   date  year  month  lwe_thickness
2024-07  2024      7     -22.030301
2024-08  2024      8     -28.221987
2024-09  2024      9     -30.288394


---
## Next

Re-run **notebook 05** (section 3 now finds `MODIS_cloud_statistics.csv`) and **notebook 06**
(sections 2b and 3 now find the melt-timing and hydrology files). Both produce the figures and
result tables; then the corresponding numbers can be written into manuscript Sections 2.5, 3.9
and 3.10, which currently carry the honest placeholders.